# TopoEyeNet — Standalone Training & Evaluation Notebook

This notebook trains **only TopoEyeNet** (not the baseline comparison models — those already ran in `topoeye-other-models.ipynb`). It fixes the two blocking issues found in the previous run:

1. **DataParallel batch-size-2 BatchNorm instability** — suspected root cause of the ~0.17–0.27 Dice ceiling. Fixed by switching to the same patch-based (CLAHE + FOV, batch size 32, single GPU) pipeline your baselines already used successfully.
2. **DRIVE external test set was being scored against the FOV mask, not the vessel ground truth** — this is why every baseline showed Specificity=0.0 and AUC=NaN on DRIVE. Fixed by pointing `DRIVE_TEST_MASK_DIR` at `test/1st_manual` instead of `test/mask`.

It trains **two ablation variants** so this one run gives you the evidence needed for the paper's novelty claim:
- `TopoEyeNet-NoConsistency` — dual mask+skeleton heads, no structural consistency loss
- `TopoEyeNet-Full` — dual heads + the structural consistency loss (the actual novelty claim)

Expect this to take a few hours (safe to run overnight). Each variant is trained, evaluated, and checkpointed to disk independently, so a mid-run interruption doesn't lose completed work.


## 1. Environment setup
Restarts the kernel after installing pinned package versions — re-run cells from #2 onward after the restart.

In [ ]:
import subprocess, os, time

def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

# Core ABI pair
run("pip uninstall -y -q numpy scipy")
run("pip install -q numpy==2.1.3 scipy==1.14.1")

# PyTorch
run("pip install -q --no-deps torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu118")

# Vision stack
run("pip install -q timm==0.9.2")
run("pip install -q segmentation-models-pytorch==0.3.3 --no-deps")
run("pip install -q albucore==0.0.16 albumentations==1.4.11 --no-deps")
run("pip install -q scikit-image==0.24.0 --no-deps")

# Utils
run("pip install -q networkx==3.3 grad-cam plotly==5.22.0 kaleido==0.2.1 wandb --no-deps")

# SMP deps + grad-cam dep
run("pip install -q munch pretrainedmodels==0.7.4 efficientnet-pytorch==0.7.1 ttach --no-deps")

# Needed for GFLOPs estimation in the evaluation harness (not in the original env-setup cell)
run("pip install -q thop --no-deps")

# Lock ABI
run("pip install -q --force-reinstall --no-deps numpy==2.1.3 scipy==1.14.1")

print("\n✅ Install complete. Restarting kernel...")
time.sleep(2)
os._exit(0)


$ pip uninstall -y -q numpy scipy
$ pip install -q numpy==2.1.3 scipy==1.14.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 45.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.


$ pip install -q --no-deps torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 108.0 MB/s eta 0:00:00
$ pip install -q timm==0.9.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 31.7 M

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cpu requires torch==2.10.0, but you have torch 2.3.0+cu118 which is incompatible.


$ pip install -q segmentation-models-pytorch==0.3.3 --no-deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 3.4 MB/s eta 0:00:00
$ pip install -q albucore==0.0.16 albumentations==1.4.11 --no-deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 5.1 MB/s eta 0:00:00
$ pip install -q scikit-image==0.24.0 --no-deps
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 96.9 MB/s eta 0:00:00
$ pip install -q networkx==3.3 grad-cam plotly==5.22.0 kaleido==0.2.1 wandb --no-deps
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 25.2 MB/s eta 0:00:00
$ pip install -q munch pretrainedmodels==0.7.4 efficientnet-pytorch==0.7.1 ttach --no-deps
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 1.9 MB/s eta 0:00:00
$ pip install

## 2. Imports & environment verification

In [1]:
import sys
print(f"Python: {sys.version}")

import numpy as np, scipy
assert np.__version__ == "2.1.3"
assert scipy.__version__ == "1.14.1"
print(f"NumPy: {np.__version__}  |  SciPy: {scipy.__version__}")

import torch, torchvision
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

import segmentation_models_pytorch as smp
print("SMP: OK")

from skimage import morphology, measure, filters
from skimage.morphology import skeletonize, medial_axis, remove_small_objects
from scipy import ndimage
from scipy.ndimage import distance_transform_edt, gaussian_filter1d
from scipy.signal import convolve2d
print("scikit-image + scipy.ndimage: OK")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import os, re, json, time, random, warnings, gc, copy
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable
import pandas as pd
from PIL import Image
import cv2
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# NOTE ON GPU USAGE: this notebook deliberately trains on a SINGLE
# GPU (no DataParallel). The original TopoEyeNet run wrapped the model
# in DataParallel with batch size 2 per GPU -- with BatchNorm layers,
# that means each GPU normalizes over a batch of only 2, which is a
# well-known source of unstable/degenerate training. The patch-based
# pipeline below trains on batches of 32+ on a single GPU, which
# removes that failure mode entirely. If you have 2 GPUs and want to
# use both, increase PATCH_BATCH_SIZE and wrap the model yourself --
# but do not go back to per-GPU batch sizes below ~16.
# ------------------------------------------------------------

device_count = torch.cuda.device_count()
print(f"\nDetected {device_count} GPU(s)")
for i in range(device_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Primary device: {device}")

def clear_gpu_cache():
    gc.collect()
    for i in range(torch.cuda.device_count()):
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device=i)

def get_gpu_memory():
    for i in range(torch.cuda.device_count()):
        a = torch.cuda.memory_allocated(i) / 1e9
        r = torch.cuda.memory_reserved(i) / 1e9
        print(f"  GPU {i}: Allocated {a:.2f} GB | Reserved {r:.2f} GB")

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

print("\nCELL 2 COMPLETE")


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.1.3  |  SciPy: 1.14.1
PyTorch: 2.3.0+cu118  |  CUDA: False
SMP: OK
scikit-image + scipy.ndimage: OK

Detected 0 GPU(s)
Primary device: cpu

CELL 2 COMPLETE


## 3. Configuration + dataset discovery (DRIVE test-mask path fixed)

In [2]:
# ============================================================
# CONFIGURATION + DATASET DISCOVERY & ANALYSIS
# ============================================================
# Same dataset-discovery/matching code as your other notebooks,
# reused unchanged. ONE FIX applied vs your previous config:
#
#   DRIVE_TEST_MASK_DIR used to point at ".../DRIVE/test/mask", which
#   in the DRIVE dataset layout is the FOV (field-of-view) mask -- a
#   solid filled disc, NOT the vessel ground truth. Evaluating against
#   it as if it were the ground truth is exactly why every model in
#   your baseline run scored Specificity=0.0 and AUC=NaN on DRIVE: the
#   "ground truth" was ~100% foreground, so any thin vessel prediction
#   looked like it was drowning in false negatives against a mask it
#   was never supposed to match. Ground truth for DRIVE test lives in
#   ".../DRIVE/test/1st_manual". Fixed below.

from dataclasses import dataclass, field
from typing import List, Tuple, Optional

@dataclass
class CFG:
    """TopoEyeNet-only Experiment Configuration"""

    EXPERIMENT_NAME: str = "topoeyenet_only_v1"
    SEED: int = 42

    RETINOMIX_IMAGE_DIR: str = "/kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/images"
    RETINOMIX_MASK_DIR: str = "/kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/mask"

    DRIVE_TRAIN_IMAGE_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/images"
    DRIVE_TRAIN_MASK_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/1st_manual"
    DRIVE_TRAIN_FOV_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/mask"

    DRIVE_TEST_IMAGE_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/images"
    # FIXED: this must be the vessel ground truth (1st_manual), not the FOV mask.
    DRIVE_TEST_MASK_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/1st_manual"
    DRIVE_TEST_FOV_DIR: str = "/kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/mask"

    OUTPUT_DIR: str = "/kaggle/working/output"
    CHECKPOINT_DIR: str = "/kaggle/working/checkpoints"
    LOG_DIR: str = "/kaggle/working/logs"
    PRECOMPUTED_DIR: str = "/kaggle/working/precomputed"
    FIGURES_DIR: str = "/kaggle/working/figures"

    IMG_SIZE: int = 512
    IN_CHANNELS: int = 3
    NUM_CLASSES: int = 1

    ENCODER: str = "resnet50"          # matches the clean TopoEyeNet architecture cell
    ENCODER_WEIGHTS: str = "imagenet"
    DECODER: str = "unet"

    USE_MASK_HEAD: bool = True
    USE_SKELETON_HEAD: bool = True

    LAMBDA_MASK: float = 1.0
    LAMBDA_SKELETON: float = 0.5
    LAMBDA_CONSISTENCY: float = 0.3

    WEIGHT_DECAY: float = 1e-5
    EARLY_STOPPING_PATIENCE: int = 8

    PRECOMPUTE_REPRESENTATIONS: bool = True
    SKELETON_METHOD: str = "zhang"
    MIN_VESSEL_SIZE: int = 20

    def create_dirs(self):
        for d in [self.OUTPUT_DIR, self.CHECKPOINT_DIR, self.LOG_DIR,
                  self.PRECOMPUTED_DIR, self.FIGURES_DIR]:
            os.makedirs(d, exist_ok=True)

    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

cfg = CFG()
cfg.create_dirs()
seed_everything(cfg.SEED)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
for k, v in cfg.to_dict().items():
    print(f"  {k}: {v}")
print("=" * 60)

# ============================================================
# DATASET DISCOVERY & MATCHING
# ============================================================

def discover_images(directory: str, extensions=('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.gif')) -> List[str]:
    if not os.path.exists(directory):
        return []
    files = []
    for root, _, filenames in os.walk(directory):
        for f in filenames:
            if f.lower().endswith(extensions):
                files.append(os.path.join(root, f))
    return sorted(files)

def get_image_info(path: str) -> dict:
    try:
        with Image.open(path) as img:
            return {'path': path, 'width': img.size[0], 'height': img.size[1],
                    'mode': img.mode, 'ok': True}
    except Exception as e:
        return {'path': path, 'width': 0, 'height': 0, 'mode': 'ERROR', 'ok': False, 'error': str(e)}

def analyze_mask(mask_path: str) -> dict:
    try:
        mask = np.array(Image.open(mask_path).convert('L'))
        if mask.max() > 1:
            mask = (mask > 127).astype(np.uint8)
        vessel_pixels = mask.sum()
        total = mask.size
        vessel_pct = 100.0 * vessel_pixels / total
        labeled = measure.label(mask, connectivity=2)
        return {
            'vessel_pixels': int(vessel_pixels), 'total_pixels': int(total),
            'vessel_pct': round(vessel_pct, 3), 'num_components': int(labeled.max()), 'ok': True
        }
    except Exception as e:
        return {'vessel_pixels': 0, 'total_pixels': 0, 'vessel_pct': 0,
                'num_components': 0, 'ok': False, 'error': str(e)}

def extract_id(path: str) -> str:
    """Source- and split-aware ID extraction (handles both RetinoMix-style
    'image'/'mask' filenames and DRIVE-original 'NN_training.tif' /
    'NN_manual1.gif' filenames)."""
    basename = os.path.splitext(os.path.basename(path))[0].lower()

    leading_match = re.match(r'^(\d+)', basename)
    if leading_match:
        return f"leading_{leading_match.group(1)}"

    key = re.sub(r'(image|mask)', '', basename)
    key = re.sub(r'_+', '_', key).strip('_')
    return key

def match_images_to_masks(image_paths, mask_paths):
    mask_lookup = {}
    for mp in mask_paths:
        mask_lookup[extract_id(mp)] = mp

    pairs, unmatched = [], []
    for ip in image_paths:
        iid = extract_id(ip)
        if iid in mask_lookup:
            pairs.append((ip, mask_lookup[iid]))
        else:
            unmatched.append(ip)

    return pairs, unmatched

class DatasetAnalyzer:
    def __init__(self, name: str, image_dir: str, mask_dir: str = None,
                 fov_dir: str = None, csv_path: str = None):
        self.name = name
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.fov_dir = fov_dir
        self.csv_path = csv_path
        self.pairs = []
        self.stats = {}

    def analyze(self):
        print(f"\n{'='*60}")
        print(f"DATASET: {self.name}")
        print(f"{'='*60}")

        image_paths = discover_images(self.image_dir)
        print(f"  Images found: {len(image_paths)}")
        if not image_paths:
            print(f"  WARNING: No images in {self.image_dir}")
            return self.stats

        sample = get_image_info(image_paths[0])
        print(f"  Sample format: {sample['mode']} | {sample['width']}x{sample['height']}")

        mask_paths = []
        if self.mask_dir and os.path.exists(self.mask_dir):
            mask_paths = discover_images(self.mask_dir)
            print(f"  Masks found: {len(mask_paths)}")
        else:
            print(f"  Masks: NOT AVAILABLE")

        if mask_paths:
            pairs, unmatched = match_images_to_masks(image_paths, mask_paths)
            self.pairs = pairs
            print(f"  Matched pairs: {len(pairs)}")
            if unmatched:
                print(f"  Unmatched images: {len(unmatched)}")
                for u in unmatched[:3]:
                    print(f"    - {os.path.basename(u)}")
        else:
            self.pairs = [(ip, None) for ip in image_paths]

        return self.stats

print("\nDiscovering datasets...")
retinomix = DatasetAnalyzer("RetinoMix-5", cfg.RETINOMIX_IMAGE_DIR, cfg.RETINOMIX_MASK_DIR)
retinomix.analyze()

drive_train = DatasetAnalyzer("DRIVE (Training)", cfg.DRIVE_TRAIN_IMAGE_DIR,
                               cfg.DRIVE_TRAIN_MASK_DIR, cfg.DRIVE_TRAIN_FOV_DIR)
drive_train.analyze()

drive_test = DatasetAnalyzer("DRIVE (Test)", cfg.DRIVE_TEST_IMAGE_DIR,
                              cfg.DRIVE_TEST_MASK_DIR, cfg.DRIVE_TEST_FOV_DIR)
drive_test.analyze()

# Sanity check on the fix: DRIVE test masks should now be sparse
# (a few % vessel pixels), NOT a filled disc (~50%+ foreground).
if drive_test.pairs:
    sample_mask_path = drive_test.pairs[0][1]
    sample_stats = analyze_mask(sample_mask_path)
    print(f"\nDRIVE test ground-truth sanity check ({os.path.basename(sample_mask_path)}):")
    print(f"  Vessel pixel %: {sample_stats['vessel_pct']:.2f}%  "
          f"(sane range for vessel GT is roughly 5-15%; if this prints "
          f">40% you are STILL reading the FOV mask, not 1st_manual)")

print("\nCELL 3 COMPLETE")


CONFIGURATION
  EXPERIMENT_NAME: topoeyenet_only_v1
  SEED: 42
  RETINOMIX_IMAGE_DIR: /kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/images
  RETINOMIX_MASK_DIR: /kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5/mask
  DRIVE_TRAIN_IMAGE_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/images
  DRIVE_TRAIN_MASK_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/1st_manual
  DRIVE_TRAIN_FOV_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/training/mask
  DRIVE_TEST_IMAGE_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/images
  DRIVE_TEST_MASK_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/test/1st_manual
  DRIVE_TEST_FOV_DIR: /kaggle/input/datasets/andrewmvd/drive-digital-retinal-images-for-vessel-extraction/DRIVE/t

## 4. Representation generator
Precomputes skeleton/distance/width/orientation/junction/endpoint targets from vessel masks.

In [3]:
# ============================================================
# CELL 4: REPRESENTATION GENERATOR
# ============================================================
# Precomputes multi-task supervision targets from vessel masks:
#   - Skeleton
#   - Distance Transform
#   - Orientation Map
#   - Junction Map
#   - Endpoint Map
#   - Width Map
# All saved to /kaggle/working/precomputed/ for fast training loading.
# Also generates comparison figures (shown inline + saved).

import os
import numpy as np
from PIL import Image
import cv2
from skimage.morphology import skeletonize, medial_axis, remove_small_objects
from scipy.ndimage import distance_transform_edt, gaussian_filter1d
from scipy import ndimage
import matplotlib.pyplot as plt
from pathlib import Path
import time

# ============================================================
# 4.1 CORE GENERATION FUNCTIONS
# ============================================================

def generate_skeleton(mask: np.ndarray, method: str = "zhang") -> np.ndarray:
    """
    Generate skeleton from binary vessel mask.
    Methods: 'zhang', 'lee', 'medial_axis'
    """
    mask_bin = (mask > 0).astype(np.uint8)
    
    if method == "zhang":
        skel = skeletonize(mask_bin, method='zhang')
    elif method == "lee":
        skel = skeletonize(mask_bin, method='lee')
    elif method == "medial_axis":
        skel, _ = medial_axis(mask_bin, return_distance=True)
    else:
        skel = skeletonize(mask_bin, method='zhang')
    
    return skel.astype(np.uint8)

def generate_distance_transform(mask: np.ndarray) -> np.ndarray:
    """
    Euclidean distance transform: each pixel = distance to nearest background.
    For vessels, this gives half-width at each point.
    """
    mask_bin = (mask > 0).astype(np.uint8)
    # Distance from vessel pixels to background
    dist = distance_transform_edt(mask_bin)
    return dist.astype(np.float32)

def generate_width_map(mask: np.ndarray, skeleton: np.ndarray) -> np.ndarray:
    """
    Vessel width at each pixel: 2 * distance transform on skeleton pixels.
    """
    dist = generate_distance_transform(mask)
    width = np.zeros_like(mask, dtype=np.float32)
    # Width = 2 * distance transform value on skeleton
    width[skeleton > 0] = 2 * dist[skeleton > 0]
    # Interpolate width to full vessel mask for dense supervision
    width = cv2.inpaint(
        (width * 255).astype(np.uint8),
        (mask == 0).astype(np.uint8),
        inpaintRadius=3,
        flags=cv2.INPAINT_NS
    )
    return width.astype(np.float32) / 255.0

def generate_orientation_map(mask: np.ndarray, skeleton: np.ndarray) -> np.ndarray:
    """
    Vessel orientation (angle in radians) at each skeleton pixel.
    Computed from local gradient of the distance transform.
    """
    dist = generate_distance_transform(mask)
    
    # Smooth distance transform
    dist_smooth = gaussian_filter1d(gaussian_filter1d(dist, sigma=1.0, axis=0), sigma=1.0, axis=1)
    
    # Compute gradient
    gy, gx = np.gradient(dist_smooth)
    
    # Orientation = angle perpendicular to gradient (along vessel)
    orientation = np.arctan2(-gx, gy)  # perpendicular to gradient
    
    # Only on skeleton
    orient_map = np.zeros_like(mask, dtype=np.float32)
    orient_map[skeleton > 0] = orientation[skeleton > 0]
    
    # Interpolate to full vessel
    if mask.sum() > 0:
        orient_norm = (orient_map + np.pi) / (2 * np.pi)  # normalize to [0,1] for inpainting
        orient_inpaint = cv2.inpaint(
            (orient_norm * 255).astype(np.uint8),
            ((mask == 0) | (skeleton == 0)).astype(np.uint8),
            inpaintRadius=5,
            flags=cv2.INPAINT_NS
        )
        orient_map = (orient_inpaint.astype(np.float32) / 255.0) * 2 * np.pi - np.pi
    
    return orient_map.astype(np.float32)

def generate_junction_map(skeleton: np.ndarray) -> np.ndarray:
    """
    Detect junctions (branching points) in skeleton.
    A junction pixel has > 2 neighbors in the skeleton.
    """
    from scipy.signal import convolve2d
    
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]], dtype=np.uint8)
    
    conv = convolve2d(skeleton, kernel, mode='same', boundary='fill')
    # Junction: center is skeleton (10) + 3+ neighbors
    junction = ((conv >= 13) & (skeleton > 0)).astype(np.uint8)
    
    # Dilate slightly for visibility
    junction = cv2.dilate(junction, np.ones((3, 3), np.uint8), iterations=1)
    
    return junction

def generate_endpoint_map(skeleton: np.ndarray) -> np.ndarray:
    """
    Detect endpoints (terminations) in skeleton.
    An endpoint has exactly 1 neighbor.
    """
    from scipy.signal import convolve2d
    
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]], dtype=np.uint8)
    
    conv = convolve2d(skeleton, kernel, mode='same', boundary='fill')
    # Endpoint: center is skeleton (10) + exactly 1 neighbor = 11
    endpoint = ((conv == 11) & (skeleton > 0)).astype(np.uint8)
    
    # Dilate for visibility
    endpoint = cv2.dilate(endpoint, np.ones((3, 3), np.uint8), iterations=1)
    
    return endpoint

def generate_all_representations(mask: np.ndarray, method: str = "zhang", min_size: int = 20) -> dict:
    """
    Generate all topological representations from a vessel mask.
    Returns dict with all maps.
    """
    # Clean mask: remove small components
    mask_clean = remove_small_objects(mask > 0, min_size=min_size)
    mask_clean = mask_clean.astype(np.uint8)
    
    # Skeleton
    skeleton = generate_skeleton(mask_clean, method=method)
    
    # Distance transform
    distance = generate_distance_transform(mask_clean)
    
    # Width map
    width = generate_width_map(mask_clean, skeleton)
    
    # Orientation
    orientation = generate_orientation_map(mask_clean, skeleton)
    
    # Junctions
    junction = generate_junction_map(skeleton)
    
    # Endpoints
    endpoint = generate_endpoint_map(skeleton)
    
    return {
        'mask': mask_clean,
        'skeleton': skeleton,
        'distance': distance,
        'width': width,
        'orientation': orientation,
        'junction': junction,
        'endpoint': endpoint,
    }

# ============================================================
# 4.2 VISUALIZATION (inline + saved)
# ============================================================

def visualize_representations(reps: dict, title: str = "Representations", 
                                 save_path: str = None, show_inline: bool = True):
    """
    Visualize all representations in a grid.
    Shows inline in notebook AND saves to file.
    """
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # Color maps for each representation
    displays = [
        ('mask', reps['mask'], 'gray', 'Vessel Mask'),
        ('skeleton', reps['skeleton'], 'gray', 'Skeleton'),
        ('distance', reps['distance'], 'viridis', 'Distance Transform'),
        ('width', reps['width'], 'plasma', 'Width Map'),
        ('orientation', (reps['orientation'] + np.pi) / (2 * np.pi), 'hsv', 'Orientation Map'),
        ('junction', reps['junction'], 'hot', 'Junction Map'),
        ('endpoint', reps['endpoint'], 'hot', 'Endpoint Map'),
    ]
    
    for ax, (key, data, cmap, label) in zip(axes, displays):
        im = ax.imshow(data, cmap=cmap)
        ax.set_title(label, fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    # Hide last subplot
    axes[-1].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    if show_inline:
        plt.show()
    else:
        plt.close()

# ============================================================
# 4.3 BATCH PROCESSING
# ============================================================

def precompute_dataset_representations(dataset: DatasetAnalyzer, 
                                        output_dir: str,
                                        method: str = "zhang",
                                        min_size: int = 20,
                                        max_samples: int = None):
    """
    Precompute representations for all masks in a dataset.
    Saves as .npz files and generates sample visualizations.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    pairs = dataset.pairs[:max_samples] if max_samples else dataset.pairs
    print(f"\nPrecomputing representations for {dataset.name}: {len(pairs)} images")
    
    stats = {
        'num_processed': 0,
        'num_junctions': [],
        'num_endpoints': [],
        'mean_width': [],
        'mean_vessel_pct': [],
    }
    
    for idx, (img_path, mask_path) in enumerate(pairs):
        if mask_path is None:
            continue
        
        # Load mask
        mask = np.array(Image.open(mask_path).convert('L'))
        if mask.max() > 1:
            mask = (mask > 127).astype(np.uint8)
        
        # Generate representations
        reps = generate_all_representations(mask, method=method, min_size=min_size)
        
        # Save as compressed NPZ
        base_name = os.path.splitext(os.path.basename(mask_path))[0]
        save_path = os.path.join(output_dir, f"{base_name}_reps.npz")
        np.savez_compressed(save_path, **reps)
        
        # Stats
        stats['num_processed'] += 1
        stats['num_junctions'].append(reps['junction'].sum())
        stats['num_endpoints'].append(reps['endpoint'].sum())
        stats['mean_width'].append(reps['width'][reps['mask'] > 0].mean())
        stats['mean_vessel_pct'].append(reps['mask'].sum() / reps['mask'].size * 100)
        
        # Visualize first 3 samples
        if idx < 3:
            fig_path = os.path.join(cfg.FIGURES_DIR, f"reps_{dataset.name.replace(' ', '_')}_{idx}.png")
            visualize_representations(
                reps, 
                title=f"{dataset.name} — {os.path.basename(mask_path)}",
                save_path=fig_path,
                show_inline=True  # Show in notebook output
            )
        
        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(pairs)}...")
    
    # Summary stats
    print(f"\n{'='*60}")
    print(f"PRECOMPUTATION SUMMARY: {dataset.name}")
    print(f"{'='*60}")
    print(f"  Processed: {stats['num_processed']}")
    print(f"  Avg junctions: {np.mean(stats['num_junctions']):.1f}")
    print(f"  Avg endpoints: {np.mean(stats['num_endpoints']):.1f}")
    print(f"  Avg width (px): {np.mean(stats['mean_width']):.2f}")
    print(f"  Avg vessel %: {np.mean(stats['mean_vessel_pct']):.2f}%")
    print(f"  Saved to: {output_dir}")
    
    return stats

# ============================================================
# 4.4 RUN ON ALL SUPERVISED DATASETS
# ============================================================

print("=" * 60)
print("REPRESENTATION GENERATOR")
print("=" * 60)

# Process RetinoMix-5
retinomix_reps_dir = os.path.join(cfg.PRECOMPUTED_DIR, "retinomix5")
retinomix_stats = precompute_dataset_representations(
    retinomix, 
    output_dir=retinomix_reps_dir,
    method=cfg.SKELETON_METHOD,
    min_size=cfg.MIN_VESSEL_SIZE
)

# Process DRIVE Training
drive_reps_dir = os.path.join(cfg.PRECOMPUTED_DIR, "drive_train")
drive_stats = precompute_dataset_representations(
    drive_train,
    output_dir=drive_reps_dir,
    method=cfg.SKELETON_METHOD,
    min_size=cfg.MIN_VESSEL_SIZE
)

# Check output size
precomputed_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk(cfg.PRECOMPUTED_DIR)
    for f in filenames if os.path.exists(os.path.join(dp, f))
) / 1e9
print(f"\nPrecomputed dir size: {precomputed_size:.3f} GB")

# Memory check
print(f"\n{'='*60}")
print("MEMORY STATUS")
print(f"{'='*60}")
get_gpu_memory()

print(f"\n{'='*60}")
print("CELL 4 COMPLETE")
print(f"{'='*60}")

REPRESENTATION GENERATOR

Precomputing representations for RetinoMix-5: 0 images

PRECOMPUTATION SUMMARY: RetinoMix-5
  Processed: 0
  Avg junctions: nan
  Avg endpoints: nan
  Avg width (px): nan
  Avg vessel %: nan%
  Saved to: /kaggle/working/precomputed/retinomix5

Precomputing representations for DRIVE (Training): 0 images

PRECOMPUTATION SUMMARY: DRIVE (Training)
  Processed: 0
  Avg junctions: nan
  Avg endpoints: nan
  Avg width (px): nan
  Avg vessel %: nan%
  Saved to: /kaggle/working/precomputed/drive_train

Precomputed dir size: 0.000 GB

MEMORY STATUS

CELL 4 COMPLETE


## 5. Train/val/test split (RetinoMix-5)

In [4]:
# ============================================================
# CELL 5: DATASET SPLIT
# ============================================================
# Splits RetinoMix-5 into train/val/test index sets and extracts the
# corresponding image/mask path lists. These lists (rm_train_images,
# rm_train_masks, rm_val_images, rm_val_masks, rm_test_images,
# rm_test_masks) and split_dataset() itself are reused unchanged by
# the benchmark suite's patch-based pipeline below.

import os
import numpy as np

# ============================================================
# 5.1 SPLIT FUNCTION
# ============================================================

def split_dataset(pairs, train_ratio=0.7, val_ratio=0.15, seed=42):
    """
    Split matched pairs into train/val/test.
    Returns indices.
    """
    np.random.seed(seed)
    n = len(pairs)
    indices = np.random.permutation(n)

    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    return train_idx, val_idx, test_idx

# Split RetinoMix-5
rm_pairs = retinomix.pairs
rm_train_idx, rm_val_idx, rm_test_idx = split_dataset(rm_pairs, train_ratio=0.7, val_ratio=0.15, seed=cfg.SEED)

print(f"\n{'='*60}")
print("RETINOMIX-5 SPLIT")
print(f"{'='*60}")
print(f"  Total: {len(rm_pairs)}")
print(f"  Train: {len(rm_train_idx)} ({len(rm_train_idx)/len(rm_pairs)*100:.1f}%)")
print(f"  Val:   {len(rm_val_idx)} ({len(rm_val_idx)/len(rm_pairs)*100:.1f}%)")
print(f"  Test:  {len(rm_test_idx)} ({len(rm_test_idx)/len(rm_pairs)*100:.1f}%)")

# Extract paths
rm_train_images = [rm_pairs[i][0] for i in rm_train_idx]
rm_train_masks = [rm_pairs[i][1] for i in rm_train_idx]
rm_val_images = [rm_pairs[i][0] for i in rm_val_idx]
rm_val_masks = [rm_pairs[i][1] for i in rm_val_idx]
rm_test_images = [rm_pairs[i][0] for i in rm_test_idx]
rm_test_masks = [rm_pairs[i][1] for i in rm_test_idx]

print(f"\n{'='*60}")
print("CELL 5 COMPLETE")
print(f"{'='*60}")



RETINOMIX-5 SPLIT
  Total: 0


ZeroDivisionError: division by zero

## 6. TopoEyeNet architecture (single GPU, no DataParallel)

In [ ]:
# ============================================================
# TOPOEYENET ARCHITECTURE
# ============================================================
# Unchanged from your original architecture cell, EXCEPT: no
# DataParallel wrapping. Training below uses patch-based batches of
# 32 on a single GPU (see the diagnostic notes in Cell 2) instead of
# whole-image batches of 2 split across 2 GPUs, which was the prime
# suspect for the Dice ceiling you saw previously.

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

class TopoEyeNet(nn.Module):
    def __init__(self,
                 encoder_name: str = "resnet50",
                 encoder_weights: str = "imagenet",
                 in_channels: int = 3,
                 mask_head: bool = True,
                 skeleton_head: bool = True):
        super().__init__()

        self.mask_head_enabled = mask_head
        self.skeleton_head_enabled = skeleton_head

        self.encoder = smp.encoders.get_encoder(
            name=encoder_name,
            in_channels=in_channels,
            depth=5,
            weights=encoder_weights,
        )

        encoder_channels = self.encoder.out_channels

        self.decoder = smp.decoders.unet.decoder.UnetDecoder(
            encoder_channels=encoder_channels,
            decoder_channels=(256, 128, 64, 32, 16),
            n_blocks=5,
            use_batchnorm=True,
            attention_type=None,
            center=False,
        )

        if mask_head:
            self.mask_head = nn.Sequential(
                nn.Conv2d(16, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(inplace=True),
                nn.Conv2d(16, 1, 1),
            )
        if skeleton_head:
            self.skeleton_head = nn.Sequential(
                nn.Conv2d(16, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(inplace=True),
                nn.Conv2d(16, 1, 1),
            )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.encoder(x)
        decoder_output = self.decoder(*features)

        outputs = {}
        if self.mask_head_enabled:
            outputs['mask'] = self.mask_head(decoder_output)
        if self.skeleton_head_enabled:
            outputs['skeleton'] = self.skeleton_head(decoder_output)

        return outputs


def build_topoeyenet(device=device):
    model = TopoEyeNet(
        encoder_name=cfg.ENCODER,
        encoder_weights=cfg.ENCODER_WEIGHTS,
        in_channels=cfg.IN_CHANNELS,
        mask_head=cfg.USE_MASK_HEAD,
        skeleton_head=cfg.USE_SKELETON_HEAD,
    ).to(device)
    return model


# Smoke test on a patch-sized input (single GPU)
print("=" * 60)
print("BUILDING TOPOEYENET (smoke test)")
print("=" * 60)

_test_model = build_topoeyenet()
total_params = sum(p.numel() for p in _test_model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model on: {next(_test_model.parameters()).device}")

_test_input = torch.randn(4, 3, 128, 128).to(device)  # patch-sized, matches PATCH_SIZE below
with torch.no_grad():
    _test_output = _test_model(_test_input)

print(f"Input: {_test_input.shape}")
for k, v in _test_output.items():
    print(f"  {k}: {v.shape}")

del _test_model, _test_input, _test_output
clear_gpu_cache()
get_gpu_memory()
print("\nARCHITECTURE CELL COMPLETE")


## 7. Loss functions (Dice, structural consistency, combined TopoEyeLoss)

In [ ]:
# ============================================================
# LOSS FUNCTIONS
# ============================================================
# 1. Dice / DiceBCE Loss       -- for mask + skeleton segmentation
# 2. Structural Consistency    -- the paper's actual novelty:
#    mask/skeleton agreement, kept exactly as you wrote it
# 3. TopoEyeLoss                -- combined multi-task loss

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# DICE / DICE+BCE
# ------------------------------------------------------------

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0, eps: float = 1e-7):
        super().__init__()
        self.smooth = smooth
        self.eps = eps

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = torch.sigmoid(pred)
        pred = pred.view(pred.size(0), -1)
        target = target.view(target.size(0), -1)

        intersection = (pred * target).sum(dim=1)
        union = pred.sum(dim=1) + target.sum(dim=1)

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth + self.eps)
        return 1.0 - dice.mean()


class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight: float = 0.7, bce_weight: float = 0.3):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.dice = DiceLoss()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        dice_loss = self.dice(pred, target)
        bce_loss = self.bce(pred, target)
        return self.dice_weight * dice_loss + self.bce_weight * bce_loss


# ------------------------------------------------------------
# STRUCTURAL CONSISTENCY (the paper's novelty claim)
# ------------------------------------------------------------

class StructuralConsistencyLoss(nn.Module):
    """A pixel predicted as 'skeleton' must also be predicted as
    'vessel' in the mask. Penalizes skeleton_prob > mask_prob."""

    def __init__(self, mode: str = "soft", eps: float = 1e-7):
        super().__init__()
        self.mode = mode
        self.eps = eps

    def forward(self, mask_pred: torch.Tensor, skeleton_pred: torch.Tensor) -> torch.Tensor:
        mask_prob = torch.sigmoid(mask_pred)
        skeleton_prob = torch.sigmoid(skeleton_pred)

        if self.mode == "soft":
            violation = F.relu(skeleton_prob - mask_prob)
            weighted_violation = violation * skeleton_prob
            consistency_loss = weighted_violation.sum() / (skeleton_prob.sum() + self.eps)
        elif self.mode == "hard":
            mask_bin = (mask_prob > 0.5).float()
            skeleton_bin = (skeleton_prob > 0.5).float()
            impossible = skeleton_bin * (1 - mask_bin)
            consistency_loss = impossible.sum() / (skeleton_bin.sum() + self.eps)

        return consistency_loss


class MaskSkeletonAgreementLoss(nn.Module):
    """Subset consistency + coverage + sparsity regularization between
    the mask and skeleton heads."""

    def __init__(self, eps: float = 1e-7):
        super().__init__()
        self.eps = eps
        self.consistency = StructuralConsistencyLoss(mode="soft")

    def forward(self, mask_pred: torch.Tensor, skeleton_pred: torch.Tensor) -> torch.Tensor:
        subset_loss = self.consistency(mask_pred, skeleton_pred)

        mask_prob = torch.sigmoid(mask_pred)
        skeleton_prob = torch.sigmoid(skeleton_pred)

        coverage = skeleton_prob * (1 - mask_prob)
        coverage_loss = coverage.sum() / (skeleton_prob.sum() + self.eps)

        sparsity_loss = skeleton_prob.mean() / (mask_prob.mean() + self.eps)
        sparsity_loss = F.relu(sparsity_loss - 0.3)

        return subset_loss + coverage_loss + 0.1 * sparsity_loss


# ------------------------------------------------------------
# COMBINED MULTI-TASK LOSS
# ------------------------------------------------------------

class TopoEyeLoss(nn.Module):
    """
    Loss = lambda_mask * DiceBCE(mask)
         + lambda_skeleton * DiceBCE(skeleton)
         + lambda_consistency * StructuralConsistency(mask, skeleton)

    Set lambda_consistency=0 to get the "no consistency" ablation
    variant (dual-head, but without the paper's actual novelty term).
    """

    def __init__(self,
                 lambda_mask: float = 1.0,
                 lambda_skeleton: float = 0.5,
                 lambda_consistency: float = 0.3):
        super().__init__()

        self.lambda_mask = lambda_mask
        self.lambda_skeleton = lambda_skeleton
        self.lambda_consistency = lambda_consistency

        self.mask_loss = DiceBCELoss(dice_weight=0.7, bce_weight=0.3)
        self.skeleton_loss = DiceBCELoss(dice_weight=0.7, bce_weight=0.3)
        self.consistency_loss = MaskSkeletonAgreementLoss()

    def forward(self, predictions: dict, targets: dict) -> dict:
        losses = {}
        total_loss = 0.0

        if 'mask' in predictions and 'mask' in targets:
            losses['mask'] = self.mask_loss(predictions['mask'], targets['mask'])
            total_loss = total_loss + self.lambda_mask * losses['mask']

        if 'skeleton' in predictions and 'skeleton' in targets:
            losses['skeleton'] = self.skeleton_loss(predictions['skeleton'], targets['skeleton'])
            total_loss = total_loss + self.lambda_skeleton * losses['skeleton']

        if self.lambda_consistency > 0 and 'mask' in predictions and 'skeleton' in predictions:
            losses['consistency'] = self.consistency_loss(predictions['mask'], predictions['skeleton'])
            total_loss = total_loss + self.lambda_consistency * losses['consistency']
        else:
            losses['consistency'] = torch.tensor(0.0, device=total_loss.device if torch.is_tensor(total_loss) else device)

        losses['total'] = total_loss
        return losses


print("Loss functions ready: DiceLoss, DiceBCELoss, StructuralConsistencyLoss, "
      "MaskSkeletonAgreementLoss, TopoEyeLoss")


## 8. Multi-task patch dataset (CLAHE + FOV, mask + skeleton targets)

In [ ]:
# ============================================================
# PATCH-BASED MULTI-TASK TRAINING PIPELINE (CLAHE + FOV masking)
# ============================================================
# Same reliable recipe your baseline models (UNet/UNet++/DeepLabV3+/
# SegFormer-B0/AttentionUNet) already used to reach 0.82-0.85 Dice:
#   1. Green channel + CLAHE contrast enhancement
#   2. FOV masking
#   3. Many overlapping patches per image, oversampling vessel-rich
#      regions, trained on a single GPU with a real batch size
#   4. At inference: overlap-tile reconstruction back to full images
#
# EXTENDED here to be multi-task: each patch also carries the aligned
# skeleton crop (from the same precomputed .npz used by the
# representation generator), so TopoEyeNet's dual heads get properly
# synchronized supervision -- not just the mask.

import numpy as np
import cv2
cv2.setNumThreads(0)  # avoid deadlock with forked DataLoader workers

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import random
import time

# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

def preprocess_fundus(image_rgb: np.ndarray) -> np.ndarray:
    green = image_rgb[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)
    enhanced = enhanced.astype(np.float32) / 255.0
    return np.stack([enhanced, enhanced, enhanced], axis=-1)


def compute_fov_mask(image_rgb: np.ndarray, threshold: int = 15) -> np.ndarray:
    green = image_rgb[:, :, 1]
    _, binary = cv2.threshold(green, threshold, 255, cv2.THRESH_BINARY)
    binary = binary.astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num_labels > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        fov = (labels == largest).astype(np.uint8)
    else:
        fov = binary

    kernel = np.ones((9, 9), np.uint8)
    fov = cv2.erode(fov, kernel, iterations=1)
    return fov


# ------------------------------------------------------------
# MULTI-TASK PATCH DATASET (mask + skeleton)
# ------------------------------------------------------------

class MultiTaskPatchDataset(Dataset):
    def __init__(self, image_paths, mask_paths, precomputed_dir,
                 patch_size=128, patches_per_image=40, vessel_oversample_ratio=0.5,
                 augment=True, warmup=True):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.precomputed_dir = precomputed_dir
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.vessel_oversample_ratio = vessel_oversample_ratio
        self.augment = augment
        self._cache = {}

        self.valid_indices = []
        for i, mp in enumerate(mask_paths):
            if mp is None:
                continue
            base = os.path.splitext(os.path.basename(mp))[0]
            npz_path = os.path.join(precomputed_dir, f"{base}_reps.npz")
            if os.path.exists(npz_path):
                self.valid_indices.append(i)

        print(f"MultiTaskPatchDataset: {len(self.valid_indices)} source images, "
              f"{patches_per_image} patches/image = "
              f"{len(self.valid_indices) * patches_per_image} patches/epoch")

        if warmup:
            self._warmup_cache()

    def _warmup_cache(self):
        print(f"  Warming up cache for {len(self.valid_indices)} images...")
        t0 = time.time()
        for n, real_idx in enumerate(self.valid_indices):
            self._load_image_and_targets(real_idx)
            if (n + 1) % 20 == 0 or (n + 1) == len(self.valid_indices):
                elapsed = time.time() - t0
                rate = (n + 1) / elapsed
                eta = (len(self.valid_indices) - (n + 1)) / rate if rate > 0 else 0
                print(f"    {n+1}/{len(self.valid_indices)} cached "
                      f"({elapsed:.1f}s elapsed, ~{eta:.1f}s remaining)")
        print(f"  Cache warmup complete: {time.time()-t0:.1f}s total")

    def __len__(self):
        return len(self.valid_indices) * self.patches_per_image

    def _load_image_and_targets(self, real_idx):
        if real_idx in self._cache:
            return self._cache[real_idx]

        img_path = self.image_paths[real_idx]
        mask_path = self.mask_paths[real_idx]

        image = np.array(Image.open(img_path).convert('RGB'))
        base = os.path.splitext(os.path.basename(mask_path))[0]
        npz_path = os.path.join(self.precomputed_dir, f"{base}_reps.npz")
        reps = dict(np.load(npz_path))
        mask = reps['mask'].astype(np.float32)
        skeleton = reps['skeleton'].astype(np.float32)

        h, w = image.shape[:2]
        if mask.shape[0] != h or mask.shape[1] != w:
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        if skeleton.shape[0] != h or skeleton.shape[1] != w:
            skeleton = cv2.resize(skeleton, (w, h), interpolation=cv2.INTER_NEAREST)

        preprocessed = preprocess_fundus(image)
        fov = compute_fov_mask(image)

        result = (preprocessed, mask, skeleton, fov)
        self._cache[real_idx] = result
        return result

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx % len(self.valid_indices)]
        image, mask, skeleton, fov = self._load_image_and_targets(real_idx)
        h, w = mask.shape
        ps = self.patch_size

        vessel_ys, vessel_xs = np.where((mask > 0) & (fov > 0))
        use_vessel_center = (
            len(vessel_ys) > 0 and random.random() < self.vessel_oversample_ratio
        )

        for _ in range(20):
            if use_vessel_center:
                i = random.randint(0, len(vessel_ys) - 1)
                cy, cx = vessel_ys[i], vessel_xs[i]
            else:
                cy = random.randint(ps // 2, h - ps // 2 - 1)
                cx = random.randint(ps // 2, w - ps // 2 - 1)

            y0 = max(0, min(h - ps, cy - ps // 2))
            x0 = max(0, min(w - ps, cx - ps // 2))

            fov_patch = fov[y0:y0 + ps, x0:x0 + ps]
            if fov_patch.mean() > 0.5:
                break

        img_patch = image[y0:y0 + ps, x0:x0 + ps].copy()
        mask_patch = mask[y0:y0 + ps, x0:x0 + ps].copy()
        skel_patch = skeleton[y0:y0 + ps, x0:x0 + ps].copy()

        if self.augment:
            if random.random() < 0.5:
                img_patch = np.fliplr(img_patch).copy()
                mask_patch = np.fliplr(mask_patch).copy()
                skel_patch = np.fliplr(skel_patch).copy()
            if random.random() < 0.5:
                img_patch = np.flipud(img_patch).copy()
                mask_patch = np.flipud(mask_patch).copy()
                skel_patch = np.flipud(skel_patch).copy()
            k = random.randint(0, 3)
            if k > 0:
                img_patch = np.rot90(img_patch, k).copy()
                mask_patch = np.rot90(mask_patch, k).copy()
                skel_patch = np.rot90(skel_patch, k).copy()

        img_tensor = torch.from_numpy(img_patch.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(mask_patch).unsqueeze(0).float()
        skel_tensor = torch.from_numpy(skel_patch).unsqueeze(0).float()

        return img_tensor, {'mask': mask_tensor, 'skeleton': skel_tensor}


# ------------------------------------------------------------
# BUILD PATCH LOADERS
# ------------------------------------------------------------

PATCH_SIZE = 128
STRIDE = 64  # 50% overlap, used at full-image inference time
PATCHES_PER_IMAGE_TRAIN = 60
PATCHES_PER_IMAGE_VAL = 20
PATCH_BATCH_SIZE = 32
PATCH_EPOCHS = 30  # matches the baseline benchmark run, for a fair comparison

print("=" * 60)
print("BUILDING MULTI-TASK PATCH DATASETS")
print("=" * 60)

patch_train_dataset = MultiTaskPatchDataset(
    rm_train_images, rm_train_masks,
    precomputed_dir=os.path.join(cfg.PRECOMPUTED_DIR, "retinomix5"),
    patch_size=PATCH_SIZE, patches_per_image=PATCHES_PER_IMAGE_TRAIN,
    vessel_oversample_ratio=0.5, augment=True, warmup=True,
)
patch_val_dataset = MultiTaskPatchDataset(
    rm_val_images, rm_val_masks,
    precomputed_dir=os.path.join(cfg.PRECOMPUTED_DIR, "retinomix5"),
    patch_size=PATCH_SIZE, patches_per_image=PATCHES_PER_IMAGE_VAL,
    vessel_oversample_ratio=0.5, augment=False, warmup=True,
)

patch_train_loader = DataLoader(
    patch_train_dataset, batch_size=PATCH_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True
)
patch_val_loader = DataLoader(
    patch_val_dataset, batch_size=PATCH_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f"Train: {len(patch_train_dataset)} patches/epoch, {len(patch_train_loader)} batches")
print(f"Val:   {len(patch_val_dataset)} patches/epoch, {len(patch_val_loader)} batches")
print(f"\nPATCH_SIZE={PATCH_SIZE}, PATCH_BATCH_SIZE={PATCH_BATCH_SIZE}, PATCH_EPOCHS={PATCH_EPOCHS}")
print("\nPATCH PIPELINE READY")


## 9. Threshold/post-processing helpers + topology metrics

In [ ]:
# ============================================================
# THRESHOLD / POST-PROCESSING HELPERS + TOPOLOGY METRICS
# ============================================================
# Unchanged from your baseline-benchmark notebook (these were already
# working correctly there -- the DRIVE bug was a data-path issue, not
# a metric-computation issue).

import numpy as np
from skimage.morphology import remove_small_objects
from skimage import measure
from scipy.signal import convolve2d


def dice_precision_recall(pred_bin, target_bin):
    tp = np.logical_and(pred_bin == 1, target_bin == 1).sum()
    fp = np.logical_and(pred_bin == 1, target_bin == 0).sum()
    fn = np.logical_and(pred_bin == 0, target_bin == 1).sum()
    eps = 1e-7
    dice = 2 * tp / (2 * tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    iou = tp / (tp + fp + fn + eps)
    return dice, precision, recall, iou


def apply_postprocess(prob_batch, threshold, min_size):
    """prob_batch: (B,1,H,W) numpy array of probabilities."""
    pred_bin = (prob_batch > threshold).astype(np.uint8)
    if min_size > 0:
        cleaned = np.zeros_like(pred_bin)
        for b in range(pred_bin.shape[0]):
            m = pred_bin[b, 0].astype(bool)
            m_clean = remove_small_objects(m, min_size=min_size)
            cleaned[b, 0] = m_clean.astype(np.uint8)
        pred_bin = cleaned
    return pred_bin


def cldice_metric(pred_skel_bin, true_mask_bin, pred_mask_bin, true_skel_bin, smooth=1e-6):
    tprec = (np.sum(pred_skel_bin * true_mask_bin) + smooth) / (np.sum(pred_skel_bin) + smooth)
    tsens = (np.sum(true_skel_bin * pred_mask_bin) + smooth) / (np.sum(true_skel_bin) + smooth)
    return 2 * tprec * tsens / (tprec + tsens + smooth)

def betti0_error(pred_skel_bin, true_skel_bin):
    """|# connected components in predicted skeleton - # in true skeleton|"""
    pred_cc = measure.label(pred_skel_bin, connectivity=2).max()
    true_cc = measure.label(true_skel_bin, connectivity=2).max()
    return abs(int(pred_cc) - int(true_cc)), int(pred_cc), int(true_cc)

def count_junctions_endpoints(skel_bin):
    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    conv = convolve2d(skel_bin.astype(np.uint8), kernel, mode='same', boundary='fill')
    junctions = ((conv >= 13) & (skel_bin > 0)).sum()
    endpoints = ((conv == 11) & (skel_bin > 0)).sum()
    return int(junctions), int(endpoints)


print("Threshold/postprocess + topology metrics ready: dice_precision_recall, "
      "apply_postprocess, cldice_metric, betti0_error, count_junctions_endpoints")


## 10. Multi-task overlap-tile inference + full-image evaluation (internal + external)

In [ ]:
# ============================================================
# MULTI-TASK OVERLAP-TILE INFERENCE + FULL-IMAGE EVALUATION
# ============================================================
# Same overlap-tile procedure your baseline models used, extended to
# pull BOTH the 'mask' and 'skeleton' head outputs from TopoEyeNet.
# Primary metrics (Dice/IoU/clDice/Betti-0/junction/endpoint) are
# computed the SAME way as your baseline table -- from the mask
# prediction, with the predicted skeleton derived via skimage
# skeletonize() -- so TopoEyeNet's numbers are directly comparable to
# Table 1/2/4 from your baseline notebook. The model's own skeleton
# HEAD prediction is reported separately as a supplementary
# "native_cldice" metric (this is the architecture-native topology
# check your Cell 16 draft described: does the skeleton head's own
# output actually land inside the mask head's own output).

import torch
import numpy as np
import cv2
from torch.cuda.amp import autocast


@torch.no_grad()
def predict_full_image_multitask(model, image_rgb, device, patch_size=PATCH_SIZE, stride=STRIDE, batch_size=32):
    """Overlap-tile prediction. Returns (mask_prob_map, skeleton_prob_map),
    both full-resolution numpy arrays in [0,1]."""
    preprocessed = preprocess_fundus(image_rgb)
    h, w = preprocessed.shape[:2]

    pad_h = (patch_size - h % stride) % stride if h % stride != 0 else 0
    pad_w = (patch_size - w % stride) % stride if w % stride != 0 else 0
    padded = np.pad(preprocessed, ((0, pad_h + patch_size), (0, pad_w + patch_size), (0, 0)), mode='reflect')
    ph, pw = padded.shape[:2]

    mask_prob_sum = np.zeros((ph, pw), dtype=np.float32)
    skel_prob_sum = np.zeros((ph, pw), dtype=np.float32)
    count = np.zeros((ph, pw), dtype=np.float32)

    coords = [(y, x) for y in range(0, ph - patch_size + 1, stride)
                      for x in range(0, pw - patch_size + 1, stride)]

    for i in range(0, len(coords), batch_size):
        batch_coords = coords[i:i + batch_size]
        batch_patches = np.stack([padded[y:y + patch_size, x:x + patch_size] for y, x in batch_coords])
        batch_tensor = torch.from_numpy(batch_patches.transpose(0, 3, 1, 2)).float().to(device)

        with autocast(enabled=(device.type == "cuda")):
            outputs = model(batch_tensor)
        mask_probs = torch.sigmoid(outputs['mask']).float().cpu().numpy()[:, 0]
        skel_probs = torch.sigmoid(outputs['skeleton']).float().cpu().numpy()[:, 0]

        for (y, x), mp, sp in zip(batch_coords, mask_probs, skel_probs):
            mask_prob_sum[y:y + patch_size, x:x + patch_size] += mp
            skel_prob_sum[y:y + patch_size, x:x + patch_size] += sp
            count[y:y + patch_size, x:x + patch_size] += 1

    mask_prob_map = mask_prob_sum / np.maximum(count, 1e-7)
    skel_prob_map = skel_prob_sum / np.maximum(count, 1e-7)
    return mask_prob_map[:h, :w], skel_prob_map[:h, :w]


def _load_mask_and_skeleton(mask_path, image_shape, precomputed_dir=None):
    """RetinoMix has precomputed reps (mask + skeleton); DRIVE does not,
    so its skeleton is generated on the fly with the same
    generate_skeleton() used to build the precomputed reps."""
    h, w = image_shape[:2]
    npz_path = None
    if precomputed_dir is not None:
        base = os.path.splitext(os.path.basename(mask_path))[0]
        candidate = os.path.join(precomputed_dir, f"{base}_reps.npz")
        if os.path.exists(candidate):
            npz_path = candidate

    if npz_path is not None:
        reps = dict(np.load(npz_path))
        mask = reps['mask'].astype(np.float32)
        skel = reps.get('skeleton')
    else:
        mask = np.array(Image.open(mask_path).convert('L')).astype(np.float32)
        mask = (mask > 127).astype(np.float32)
        skel = None

    if mask.shape[0] != h or mask.shape[1] != w:
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    mask_bin = (mask > 0.5).astype(np.uint8)
    if skel is None:
        skel_bin = generate_skeleton(mask_bin, method=cfg.SKELETON_METHOD).astype(np.uint8)
    else:
        if skel.shape[0] != h or skel.shape[1] != w:
            skel = cv2.resize(skel.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        skel_bin = (skel > 0).astype(np.uint8)

    return mask_bin, skel_bin


@torch.no_grad()
def evaluate_full_image_dataset_multitask(model, device, image_mask_pairs, threshold=0.5, min_size=0,
                                           precomputed_dir=None, apply_fov=True, dataset_name="",
                                           skeleton_threshold=0.5):
    """Full-image evaluation via overlap-tile inference for TopoEyeNet.
    Returns (summary_metrics_dict, per_image_records_list). Primary
    metrics use the MASK head (comparable to baseline Table 1/2/4);
    'native_cldice' additionally checks the skeleton HEAD's own output
    against the mask HEAD's own output (architecture-native topology
    consistency, not just the standard clDice)."""
    model.eval()
    tp = fp = fn = tn = 0
    probs_sample, targets_sample = [], []
    per_image = []
    native_cldice_scores = []

    for idx, (img_path, mask_path) in enumerate(image_mask_pairs):
        image = np.array(Image.open(img_path).convert('RGB'))
        mask_bin, true_skel = _load_mask_and_skeleton(mask_path, image.shape, precomputed_dir)

        fov = compute_fov_mask(image) if apply_fov else np.ones(mask_bin.shape, dtype=np.uint8)

        mask_prob_map, skel_prob_map = predict_full_image_multitask(model, image, device)
        pred_bin = (mask_prob_map > threshold).astype(np.uint8)
        if min_size > 0:
            pred_bin = remove_small_objects(pred_bin.astype(bool), min_size=min_size).astype(np.uint8)
        pred_bin = pred_bin * fov

        pred_skel_head_bin = (skel_prob_map > skeleton_threshold).astype(np.uint8) * fov

        valid = fov > 0
        tp_i = np.logical_and(pred_bin[valid] == 1, mask_bin[valid] == 1).sum()
        fp_i = np.logical_and(pred_bin[valid] == 1, mask_bin[valid] == 0).sum()
        fn_i = np.logical_and(pred_bin[valid] == 0, mask_bin[valid] == 1).sum()
        tn_i = np.logical_and(pred_bin[valid] == 0, mask_bin[valid] == 0).sum()
        tp += tp_i; fp += fp_i; fn += fn_i; tn += tn_i

        probs_sample.append(mask_prob_map[valid][::10])
        targets_sample.append(mask_bin[valid][::10])

        # --- standard topology metrics (skeleton derived from mask prediction,
        #     same procedure as the baseline table -> directly comparable) ---
        pred_skel = skeletonize(pred_bin.astype(bool)).astype(np.uint8)
        cld = cldice_metric(pred_skel, mask_bin, pred_bin, true_skel)
        b0_err, pred_cc, true_cc = betti0_error(pred_skel, true_skel)
        pred_j, pred_e = count_junctions_endpoints(pred_skel)
        true_j, true_e = count_junctions_endpoints(true_skel)

        # --- architecture-native check: does the skeleton HEAD's own
        #     prediction land inside the mask HEAD's own prediction? ---
        native_cld = cldice_metric(pred_skel_head_bin, mask_bin, pred_bin, true_skel)
        native_cldice_scores.append(native_cld)

        eps = 1e-7
        dice_i = 2 * tp_i / (2 * tp_i + fp_i + fn_i + eps)
        per_image.append(dict(
            image=os.path.basename(img_path), dice=dice_i, cldice=cld,
            betti0_error=b0_err, junction_ae=abs(pred_j - true_j), endpoint_ae=abs(pred_e - true_e),
            native_cldice=native_cld,
        ))

        if (idx + 1) % 10 == 0 or (idx + 1) == len(image_mask_pairs):
            print(f"    [{dataset_name}] {idx + 1}/{len(image_mask_pairs)} images evaluated")

    eps = 1e-7
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    sensitivity = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    precision = tp / (tp + fp + eps)
    dice = 2 * tp / (2 * tp + fp + fn + eps)
    iou = tp / (tp + fp + fn + eps)

    all_probs = np.concatenate(probs_sample)
    all_targets = np.concatenate(targets_sample)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except Exception:
        auc = float('nan')

    per_df = pd.DataFrame(per_image)
    summary = dict(
        accuracy=accuracy, precision=precision, recall=sensitivity, specificity=specificity,
        dice=dice, iou=iou, auc_roc=auc,
        cldice=per_df['cldice'].mean(), betti0_error=per_df['betti0_error'].mean(),
        junction_mae=per_df['junction_ae'].mean(), endpoint_mae=per_df['endpoint_ae'].mean(),
        native_cldice=float(np.mean(native_cldice_scores)),
        confusion_matrix=dict(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn)),
    )
    return summary, per_image


def optimize_threshold_on_val_multitask(model, device, val_loader, threshold_grid, min_size_grid):
    """Grid-search threshold/min_size on VAL, maximizing MASK-head Dice.
    Never touches test."""
    model.eval()
    val_probs, val_targets = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            mask_target = targets['mask']
            with autocast(enabled=(device.type == "cuda")):
                outputs = model(images)
            val_probs.append(torch.sigmoid(outputs['mask']).float().cpu().numpy())
            val_targets.append(mask_target.numpy())

    best_dice, best_config = -1, (0.5, 0)
    for thresh in threshold_grid:
        for min_size in min_size_grid:
            tp = fp = fn = 0
            for prob_batch, target_batch in zip(val_probs, val_targets):
                pred_bin = apply_postprocess(prob_batch, thresh, min_size)
                target_bin = (target_batch > 0.5).astype(np.uint8)
                tp += np.logical_and(pred_bin == 1, target_bin == 1).sum()
                fp += np.logical_and(pred_bin == 1, target_bin == 0).sum()
                fn += np.logical_and(pred_bin == 0, target_bin == 1).sum()
            dice = 2 * tp / (2 * tp + fp + fn + 1e-7)
            if dice > best_dice:
                best_dice, best_config = dice, (thresh, min_size)

    return best_config, best_dice


def count_params_millions(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def estimate_gflops(model, device, patch_size=PATCH_SIZE):
    try:
        from thop import profile
        dummy = torch.randn(1, 3, patch_size, patch_size).to(device)

        class _MaskOnlyWrapper(nn.Module):
            """thop profiles a single-tensor-output module more reliably
            than one returning a dict."""
            def __init__(self, m):
                super().__init__()
                self.m = m
            def forward(self, x):
                return self.m(x)['mask']

        wrapped = _MaskOnlyWrapper(copy.deepcopy(model)).to(device)
        macs, _ = profile(wrapped, inputs=(dummy,), verbose=False)
        return macs * 2 / 1e9  # MACs -> FLOPs
    except Exception as e:
        print(f"    (GFLOPs estimation skipped: {e})")
        return float('nan')


@torch.no_grad()
def benchmark_inference_speed(model, device, patch_size=PATCH_SIZE, n_warmup=5, n_iters=30):
    model.eval()
    dummy = torch.randn(1, 3, patch_size, patch_size).to(device)
    for _ in range(n_warmup):
        _ = model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iters):
        _ = model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    per_image_ms = (elapsed / n_iters) * 1000
    fps = 1000.0 / per_image_ms
    return per_image_ms, fps


print("Multi-task evaluation utilities ready: predict_full_image_multitask, "
      "evaluate_full_image_dataset_multitask, optimize_threshold_on_val_multitask, "
      "count_params_millions, estimate_gflops, benchmark_inference_speed")


## 11. Grad-CAM / faithfulness / TCAV utilities (reused, already multi-task aware)

In [ ]:
# ============================================================
# PORTABLE MODEL BENCHMARK REPORT
# ============================================================
# Self-contained: designed to be copy-pasted into a DIFFERENT notebook
# for comparing multiple architectures against each other. Only needs:
# a model, a test dataloader, a device. Produces one row per model ->
# stack rows into a comparison
# DataFrame across all models you benchmark.
#
# Includes:
#   1. Standard segmentation metrics (Dice, IoU, Accuracy, Sensitivity,
#      Specificity, Precision, AUC-ROC) -- accumulated confusion-matrix
#      style across the full test set (correct for imbalanced classes,
#      unlike per-batch-averaged Dice).
#   2. Grad-CAM adapted for segmentation ("Seg-Grad-CAM") -- the
#      gradient-based saliency method. Target is the sum of predicted
#      foreground logits, so the CAM highlights what the encoder
#      "looked at" to produce the vessel prediction.
#   3. Deletion-AUC faithfulness score -- a QUANTITATIVE XAI metric,
#      not just a picture: progressively occludes the most-salient
#      pixels (per Grad-CAM) and measures how fast the prediction
#      degrades. Steeper/faster degradation = more faithful saliency
#      map = the model is "really" using what Grad-CAM says it's using.
#      Reported as a single number so it belongs in a metrics table.
#   4. TCAV (Testing with Concept Activation Vectors) -- requires you
#      to supply small folders of concept-positive and concept-negative
#      example images (e.g. "thin vessel crops" vs "random background
#      crops", or "junction crops" vs "non-junction crops"). Reports
#      what fraction of target images the model's decision is
#      positively sensitive to that concept.
#
# Usage pattern for a multi-model comparison notebook:
#
#   rows = []
#   for name, model in {"UNet": unet_model, "UNet++": unetpp_model,
#                        "DeepLabV3+": deeplab_model}.items():
#       row = full_benchmark_report(
#           model_name=name, model=model, test_loader=test_loader,
#           device=device, is_multitask=False,
#           target_layer=get_last_conv_layer(model),
#       )
#       rows.append(row)
#   comparison_df = pd.DataFrame(rows)
#   comparison_df.to_csv("model_comparison.csv", index=False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
import pandas as pd


# ============================================================
# 1. SEGMENTATION METRICS (accumulated, imbalance-correct)
# ============================================================

@torch.no_grad()
def compute_segmentation_metrics(model, test_loader, device, is_multitask=False,
                                  mask_key='mask', threshold=0.5, auc_subsample=4):
    """
    is_multitask: True if model(images) returns a dict with a 'mask'
    key; False if model(images) returns a plain logits tensor (like a
    standard smp.Unet).
    """
    model.eval()
    tp = fp = tn = fn = 0
    probs_sample, targets_sample = [], []

    for images, targets in test_loader:
        images = images.to(device, non_blocking=True)
        mask_target = (targets[mask_key] if isinstance(targets, dict) else targets).to(device, non_blocking=True)

        outputs = model(images)
        logits = outputs['mask'] if is_multitask else outputs
        probs = torch.sigmoid(logits)
        preds = (probs > threshold).float()

        tp += ((preds == 1) & (mask_target == 1)).sum().item()
        tn += ((preds == 0) & (mask_target == 0)).sum().item()
        fp += ((preds == 1) & (mask_target == 0)).sum().item()
        fn += ((preds == 0) & (mask_target == 1)).sum().item()

        probs_sample.append(probs.flatten()[::auc_subsample].cpu().numpy())
        targets_sample.append(mask_target.flatten()[::auc_subsample].cpu().numpy())

    eps = 1e-7
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    sensitivity = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    precision = tp / (tp + fp + eps)
    dice = 2 * tp / (2 * tp + fp + fn + eps)
    iou = tp / (tp + fp + fn + eps)

    all_probs = np.concatenate(probs_sample)
    all_targets = np.concatenate(targets_sample)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except Exception:
        auc = float('nan')

    return {
        'accuracy': accuracy, 'auc_roc': auc, 'specificity': specificity,
        'sensitivity': sensitivity, 'precision': precision,
        'dice': dice, 'iou': iou,
    }


# ============================================================
# 2. SEG-GRAD-CAM
# ============================================================

class SegGradCAM:
    """
    Grad-CAM adapted for dense segmentation output. Hooks a chosen
    conv layer (typically the last encoder block, or a late decoder
    block), backprops from the SUM of predicted-foreground logits,
    and produces a spatial heatmap of "what the model used."
    """

    def __init__(self, model, target_layer, is_multitask=False):
        self.model = model
        self.is_multitask = is_multitask
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, image_tensor):
        """image_tensor: (1, C, H, W). Returns CAM as (H, W) numpy array in [0,1]."""
        self.model.zero_grad()
        image_tensor = image_tensor.clone().requires_grad_(True)
        outputs = self.model(image_tensor)
        logits = outputs['mask'] if self.is_multitask else outputs

        target = torch.sigmoid(logits).sum()
        target.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # global-avg-pool of gradients
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=image_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-7)
        return cam


def get_last_conv_layer(model):
    """
    Convenience helper: finds a meaningful FEATURE layer for Grad-CAM.
    IMPORTANT: the naive "last Conv2d in the model" is almost always
    the final 1-channel output/classifier head in a segmentation model
    (e.g. smp.Unet's segmentation_head), which produces a degenerate,
    uninformative CAM. This instead returns the SECOND-to-last conv
    layer, skipping the output head, which is a much more meaningful
    default target for Grad-CAM. For best results, still consider
    passing target_layer explicitly (e.g. the last block of the decoder).
    """
    conv_layers = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    if len(conv_layers) == 0:
        raise ValueError("No Conv2d layer found -- pass target_layer explicitly.")
    if len(conv_layers) == 1:
        return conv_layers[0]
    return conv_layers[-2]  # skip the final output/head conv


# ============================================================
# 3. DELETION-AUC FAITHFULNESS SCORE
# ============================================================

@torch.no_grad()
def deletion_auc(model, image_tensor, cam, device, is_multitask=False, steps=10):
    """
    Progressively occludes the top-CAM% of pixels (most salient first)
    and measures how the mean predicted foreground probability drops.
    Faithful saliency -> probability drops FAST as salient pixels are
    removed -> SMALL area under the curve. Returns 1 - normalized_AUC,
    so HIGHER score = MORE faithful (more intuitive for a comparison
    table where "higher is better" everywhere).
    """
    model.eval()
    h, w = cam.shape
    flat_order = np.argsort(-cam.flatten())  # most salient first

    fractions = np.linspace(0, 1, steps)
    probs_at_fraction = []

    img = image_tensor.clone()
    baseline_value = img.mean()  # occlude with mean pixel value

    for frac in fractions:
        n_pixels = int(frac * len(flat_order))
        occluded = image_tensor.clone()
        if n_pixels > 0:
            idx = flat_order[:n_pixels]
            ys, xs = np.unravel_index(idx, (h, w))
            occluded[0, :, ys, xs] = baseline_value

        outputs = model(occluded.to(device))
        logits = outputs['mask'] if is_multitask else outputs
        mean_prob = torch.sigmoid(logits).mean().item()
        probs_at_fraction.append(mean_prob)

    probs_at_fraction = np.array(probs_at_fraction)
    # Normalize so AUC is comparable across images/models.
    # np.trapz was removed in NumPy 2.x (renamed np.trapezoid) -- use a
    # manual trapezoidal rule so this works regardless of NumPy version.
    auc_raw = np.sum((probs_at_fraction[:-1] + probs_at_fraction[1:]) / 2 * np.diff(fractions))
    auc = auc_raw / (probs_at_fraction[0] + 1e-7)
    faithfulness_score = 1.0 - auc  # higher = better (faster degradation)
    return faithfulness_score


# ============================================================
# 4. TCAV (Testing with Concept Activation Vectors)
# ============================================================

def extract_activations(model, target_layer, images, device):
    """Returns flattened (global-avg-pooled) activations at target_layer
    for a batch of images, shape (N, C). Each item in `images` can be
    either (C,H,W) unbatched or (1,C,H,W) already-batched."""
    activations = []

    def hook(module, input, output):
        activations.append(output.detach().mean(dim=(2, 3)).cpu().numpy())

    handle = target_layer.register_forward_hook(hook)
    model.eval()
    with torch.no_grad():
        for img in images:
            img_batched = img.unsqueeze(0) if img.dim() == 3 else img
            model(img_batched.to(device))
    handle.remove()
    return np.concatenate(activations, axis=0)


def compute_tcav_score(model, target_layer, concept_positive_images, concept_negative_images,
                        target_images, device, is_multitask=False):
    """
    Standard TCAV algorithm:
      1. Extract activations at target_layer for concept-positive and
         concept-negative example images.
      2. Train a linear classifier to separate them -> the CAV
         (Concept Activation Vector) is the direction perpendicular
         to that classifier's decision boundary.
      3. For each target image, compute the directional derivative of
         the model's output along the CAV direction.
      4. TCAV score = fraction of target images where that directional
         derivative is positive (i.e. the concept increases the
         foreground prediction).

    concept_positive_images / concept_negative_images / target_images:
    lists of (1, C, H, W) image tensors (already preprocessed/normalized).
    """
    from sklearn.linear_model import LogisticRegression

    pos_acts = extract_activations(model, target_layer, concept_positive_images, device)
    neg_acts = extract_activations(model, target_layer, concept_negative_images, device)

    X = np.concatenate([pos_acts, neg_acts], axis=0)
    y = np.concatenate([np.ones(len(pos_acts)), np.zeros(len(neg_acts))])

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y)
    cav = clf.coef_[0]
    cav = cav / (np.linalg.norm(cav) + 1e-7)

    sensitivities = []
    cav_tensor = torch.from_numpy(cav).float().to(device)

    for img in target_images:
        img = img.clone().to(device).requires_grad_(True)
        activation_holder = {}

        def hook(module, input, output):
            activation_holder['act'] = output

        handle = target_layer.register_forward_hook(hook)
        outputs = model(img.unsqueeze(0) if img.dim() == 3 else img)
        logits = outputs['mask'] if is_multitask else outputs
        target = torch.sigmoid(logits).sum()

        act = activation_holder['act']
        pooled_act = act.mean(dim=(2, 3)).squeeze()
        grad = torch.autograd.grad(target, act, retain_graph=False)[0]
        grad_pooled = grad.mean(dim=(2, 3)).squeeze()

        directional_derivative = torch.dot(grad_pooled, cav_tensor).item()
        sensitivities.append(directional_derivative)
        handle.remove()

    sensitivities = np.array(sensitivities)
    tcav_score = (sensitivities > 0).mean()
    return tcav_score, sensitivities


# ============================================================
# 5. FULL BENCHMARK REPORT (ties everything together)
# ============================================================

def full_benchmark_report(model_name, model, test_loader, device, is_multitask=False,
                           target_layer=None, concept_pos=None, concept_neg=None,
                           n_gradcam_samples=5):
    """
    Runs everything and returns ONE FLAT DICT (a row) suitable for
    stacking into a pandas DataFrame across multiple models.
    """
    print(f"\n{'='*60}")
    print(f"BENCHMARKING: {model_name}")
    print(f"{'='*60}")

    row = {'model': model_name}

    # --- Segmentation metrics ---
    print("  Computing segmentation metrics...")
    seg_metrics = compute_segmentation_metrics(model, test_loader, device, is_multitask=is_multitask)
    row.update(seg_metrics)
    print(f"    Dice: {seg_metrics['dice']:.4f} | IoU: {seg_metrics['iou']:.4f} | "
          f"AUC: {seg_metrics['auc_roc']:.4f}")

    # --- Grad-CAM faithfulness (averaged over a few test images) ---
    if target_layer is None:
        target_layer = get_last_conv_layer(model)

    print(f"  Computing Grad-CAM faithfulness over {n_gradcam_samples} sample images...")
    cam_extractor = SegGradCAM(model, target_layer, is_multitask=is_multitask)
    faithfulness_scores = []
    sample_count = 0
    for images, _ in test_loader:
        for i in range(images.shape[0]):
            if sample_count >= n_gradcam_samples:
                break
            img = images[i:i+1].to(device)
            cam = cam_extractor(img)
            score = deletion_auc(model, img, cam, device, is_multitask=is_multitask)
            faithfulness_scores.append(score)
            sample_count += 1
        if sample_count >= n_gradcam_samples:
            break
    row['gradcam_faithfulness'] = float(np.mean(faithfulness_scores))
    print(f"    Grad-CAM faithfulness (Deletion-AUC based): {row['gradcam_faithfulness']:.4f}")

    # --- TCAV (optional -- only if concept examples supplied) ---
    if concept_pos is not None and concept_neg is not None:
        print("  Computing TCAV score...")
        target_images = [images[i:i+1] for images, _ in test_loader for i in range(images.shape[0])][:20]
        tcav_score, _ = compute_tcav_score(
            model, target_layer, concept_pos, concept_neg, target_images, device,
            is_multitask=is_multitask
        )
        row['tcav_score'] = tcav_score
        print(f"    TCAV score: {tcav_score:.4f}")
    else:
        row['tcav_score'] = None
        print("  Skipping TCAV (no concept_pos/concept_neg example images supplied)")

    return row




## 12. Training harness for the two ablation variants

In [ ]:
# ============================================================
# TRAINING HARNESS FOR TOPOEYENET VARIANTS (ABLATION)
# ============================================================
# Trains TWO variants back-to-back so this single overnight run
# produces the ablation evidence, not just one number:
#
#   1. "TopoEyeNet-NoConsistency" : dual mask+skeleton heads,
#      lambda_consistency = 0  -> isolates what the extra skeleton
#      head alone buys you over a plain mask-only U-Net (your existing
#      "UNet" baseline from the other notebook already covers the
#      mask-only case, so it is not re-trained here).
#   2. "TopoEyeNet-Full"         : dual heads + structural consistency
#      loss (lambda_consistency = cfg.LAMBDA_CONSISTENCY) -> the full
#      paper claim.
#
# Same hyperparameters as your baseline benchmark run (30 epochs,
# AdamW, cosine LR, patch batch 32) so results are directly comparable.

BENCHMARK_ROOT = os.path.join(cfg.OUTPUT_DIR, "benchmarks")
os.makedirs(BENCHMARK_ROOT, exist_ok=True)


def benchmark_dirs(name: str) -> dict:
    base = os.path.join(BENCHMARK_ROOT, name)
    sub = {k: os.path.join(base, k) for k in
           ["weights", "logs", "history", "predictions", "metrics", "gradcam", "confusion_matrices"]}
    for d in sub.values():
        os.makedirs(d, exist_ok=True)
    return sub


BENCH_CFG = dict(
    epochs=PATCH_EPOCHS,
    lr=1e-4,
    weight_decay=cfg.WEIGHT_DECAY,
    eta_min=1e-6,
    early_stopping_patience=cfg.EARLY_STOPPING_PATIENCE,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    batch_size=PATCH_BATCH_SIZE,
    threshold_grid=[0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    min_size_grid=[0, 15, 30, 50],
)

BENCH_DEVICE = device

print("=" * 60)
print("TOPOEYENET ABLATION SUITE INITIALIZED")
print("=" * 60)
print(f"Variants to train: TopoEyeNet-NoConsistency, TopoEyeNet-Full")
print(f"Output root:       {BENCHMARK_ROOT}")
print(f"Device:             {BENCH_DEVICE}")
for k, v in BENCH_CFG.items():
    print(f"  {k}: {v}")


def dice_from_logits_bench(logits, target, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    tp = (preds * target).sum()
    fp = (preds * (1 - target)).sum()
    fn = ((1 - preds) * target).sum()
    return (2 * tp / (2 * tp + fp + fn + 1e-7)).item()


def train_topoeye_variant(name: str, lambda_consistency: float, device=BENCH_DEVICE, verbose=True):
    """Trains one TopoEyeNet variant. lambda_consistency=0 disables the
    structural-consistency term for the ablation. Returns the trained
    model (best-val-Dice checkpoint loaded) and its history dataframe."""
    dirs = benchmark_dirs(name)
    print(f"\n{'=' * 60}")
    print(f"TRAINING: {name}  (lambda_consistency={lambda_consistency})")
    print(f"{'=' * 60}")

    model = build_topoeyenet(device=device)
    criterion = TopoEyeLoss(
        lambda_mask=cfg.LAMBDA_MASK,
        lambda_skeleton=cfg.LAMBDA_SKELETON,
        lambda_consistency=lambda_consistency,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=BENCH_CFG["lr"], weight_decay=BENCH_CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=BENCH_CFG["epochs"], eta_min=BENCH_CFG["eta_min"])
    scaler = GradScaler(enabled=(device.type == "cuda"))

    best_val_dice = 0.0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, BENCH_CFG["epochs"] + 1):
        model.train()
        t0 = time.time()
        train_loss, train_dice, n = 0.0, 0.0, 0

        for images, targets in patch_train_loader:
            images = images.to(device, non_blocking=True)
            mask_t = targets['mask'].to(device, non_blocking=True)
            skel_t = targets['skeleton'].to(device, non_blocking=True)

            with autocast(enabled=(device.type == "cuda")):
                outputs = model(images)
                loss_dict = criterion(outputs, {'mask': mask_t, 'skeleton': skel_t})
                loss = loss_dict['total']

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()
            train_dice += dice_from_logits_bench(outputs['mask'], mask_t)
            n += 1

        scheduler.step()
        train_loss /= n
        train_dice /= n

        model.eval()
        val_dice, n_val = 0.0, 0
        with torch.no_grad():
            for images, targets in patch_val_loader:
                images = images.to(device, non_blocking=True)
                mask_t = targets['mask'].to(device, non_blocking=True)
                with autocast(enabled=(device.type == "cuda")):
                    outputs = model(images)
                val_dice += dice_from_logits_bench(outputs['mask'], mask_t)
                n_val += 1
        val_dice /= n_val

        improved = val_dice > best_val_dice
        if improved:
            best_val_dice = val_dice
            epochs_without_improvement = 0
            torch.save(model.state_dict(), os.path.join(dirs["weights"], f"{name}_best.pt"))
        else:
            epochs_without_improvement += 1

        elapsed = time.time() - t0
        history.append(dict(epoch=epoch, train_loss=train_loss, train_dice=train_dice,
                             val_dice=val_dice, lr=optimizer.param_groups[0]['lr'],
                             epoch_time_s=elapsed, improved=improved))
        if verbose:
            print(f"[{name}] Epoch {epoch}/{BENCH_CFG['epochs']} | Time {elapsed:.1f}s | "
                  f"Train Loss {train_loss:.4f} Dice {train_dice:.4f} | "
                  f"Val Dice {val_dice:.4f} | LR {optimizer.param_groups[0]['lr']:.2e}"
                  + ("  * new best" if improved else ""))

        if epochs_without_improvement >= BENCH_CFG["early_stopping_patience"]:
            print(f"[{name}] Early stopping at epoch {epoch} "
                  f"(no improvement for {BENCH_CFG['early_stopping_patience']} epochs)")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(dirs["history"], f"{name}_history.csv"), index=False)
    with open(os.path.join(dirs["logs"], f"{name}_train_log.json"), "w") as f:
        json.dump(dict(model=name, lambda_consistency=lambda_consistency,
                        best_val_dice=best_val_dice, hyperparameters=BENCH_CFG), f, indent=2)

    model.load_state_dict(torch.load(os.path.join(dirs["weights"], f"{name}_best.pt"), map_location=device))
    model.eval()

    print(f"[{name}] Best val Dice: {best_val_dice:.4f}  ->  saved to "
          f"{dirs['weights']}/{name}_best.pt")
    return model, history_df


print("\nTraining harness ready: train_topoeye_variant(name, lambda_consistency)")


## 13. Full per-variant evaluation wiring

In [ ]:
# ============================================================
# FULL PER-VARIANT EVALUATION
# ============================================================
# Runs every evaluation for ONE trained TopoEyeNet variant and saves
# artifacts under outputs/benchmarks/<name>/. Same 6-step structure as
# your baseline notebook so the resulting metrics.json is a drop-in
# row alongside UNet/UNet++/DeepLabV3+/SegFormer-B0/AttentionUNet.

def evaluate_topoeye_variant(name: str, model, device=BENCH_DEVICE):
    dirs = benchmark_dirs(name)
    print(f"\n{'=' * 60}")
    print(f"EVALUATING: {name}")
    print(f"{'=' * 60}")

    # --- 1. Threshold + post-processing optimization on VAL ---
    print("  [1/6] Threshold/post-processing grid search on VAL...")
    (best_thresh, best_min_size), val_dice = optimize_threshold_on_val_multitask(
        model, device, patch_val_loader, BENCH_CFG["threshold_grid"], BENCH_CFG["min_size_grid"])
    print(f"        Best config: threshold={best_thresh}, min_size={best_min_size} (val Dice={val_dice:.4f})")

    # --- 2. Internal test set: RETINOMIX-5 (full-image, overlap-tile, FOV-masked) ---
    print("  [2/6] Internal RETINOMIX-5 test evaluation (full-image)...")
    internal_pairs = list(zip(rm_test_images, rm_test_masks))
    internal_metrics, internal_per_image = evaluate_full_image_dataset_multitask(
        model, device, internal_pairs, threshold=best_thresh, min_size=best_min_size,
        precomputed_dir=os.path.join(cfg.PRECOMPUTED_DIR, "retinomix5"),
        apply_fov=True, dataset_name=f"{name}/RETINOMIX-5",
    )
    pd.DataFrame(internal_per_image).to_csv(
        os.path.join(dirs["predictions"], f"{name}_retinomix5_test_per_image.csv"), index=False)

    # --- 3. External test set: DRIVE (cross-dataset, no retraining) ---
    #     Uses cfg.DRIVE_TEST_MASK_DIR, which is now fixed to 1st_manual
    #     (real vessel ground truth) instead of the FOV mask.
    print("  [3/6] External DRIVE test evaluation (cross-dataset, no retraining)...")
    drive_image_paths = discover_images(cfg.DRIVE_TEST_IMAGE_DIR)
    drive_mask_paths = discover_images(cfg.DRIVE_TEST_MASK_DIR)
    drive_pairs, _ = match_images_to_masks(drive_image_paths, drive_mask_paths)
    if drive_pairs:
        external_metrics, external_per_image = evaluate_full_image_dataset_multitask(
            model, device, drive_pairs, threshold=best_thresh, min_size=best_min_size,
            precomputed_dir=None, apply_fov=True, dataset_name=f"{name}/DRIVE",
        )
        pd.DataFrame(external_per_image).to_csv(
            os.path.join(dirs["predictions"], f"{name}_drive_test_per_image.csv"), index=False)
    else:
        print("        WARNING: no matched DRIVE test pairs found -- skipping external eval.")
        external_metrics = {k: float('nan') for k in internal_metrics}

    # --- 4. Efficiency: parameters, GFLOPs, inference time, FPS ---
    print("  [4/6] Efficiency benchmarking...")
    n_params_m = count_params_millions(model)
    gflops = estimate_gflops(model, device)
    inf_ms, fps = benchmark_inference_speed(model, device)
    efficiency_metrics = dict(parameters_M=n_params_m, gflops=gflops,
                               inference_time_ms=inf_ms, fps=fps)
    print(f"        Params: {n_params_m:.2f}M | GFLOPs: {gflops:.2f} | "
          f"Inference: {inf_ms:.2f} ms/patch | FPS: {fps:.1f}")

    # --- 5. Confusion matrix figure (internal test set) ---
    print("  [5/6] Saving confusion matrix...")
    cm = internal_metrics["confusion_matrix"]
    cm_arr = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm_arr, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm_arr):
        ax.text(j, i, f"{v:,}", ha="center", va="center",
                color="white" if v > cm_arr.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["True 0", "True 1"])
    ax.set_title(f"{name} — Confusion Matrix (RETINOMIX-5 test)")
    plt.tight_layout()
    plt.savefig(os.path.join(dirs["confusion_matrices"], f"{name}_confusion_matrix.png"), dpi=150)
    plt.close()

    # --- 6. Grad-CAM: heatmaps, overlays, faithfulness ---
    print("  [6/6] Grad-CAM generation...")
    try:
        target_layer = get_last_conv_layer(model)
        cam_extractor = SegGradCAM(model, target_layer, is_multitask=True)
        n_samples = min(5, len(internal_pairs))
        faithfulness_scores = []
        fig, axes = plt.subplots(n_samples, 3, figsize=(9, 3 * n_samples))
        if n_samples == 1:
            axes = axes.reshape(1, -1)
        for i in range(n_samples):
            img_path, _ = internal_pairs[i]
            image_rgb = np.array(Image.open(img_path).convert('RGB'))
            preproc = preprocess_fundus(image_rgb)
            ph = pw = BENCH_CFG["patch_size"]
            crop = cv2.resize(preproc, (pw, ph))
            img_tensor = torch.from_numpy(crop.transpose(2, 0, 1)).float().unsqueeze(0).to(device)

            cam = cam_extractor(img_tensor)
            score = deletion_auc(model, img_tensor, cam, device, is_multitask=True)
            faithfulness_scores.append(score)

            disp_img = crop
            axes[i, 0].imshow(disp_img); axes[i, 0].set_title("Input"); axes[i, 0].axis('off')
            axes[i, 1].imshow(cam, cmap='jet'); axes[i, 1].set_title("Grad-CAM"); axes[i, 1].axis('off')
            axes[i, 2].imshow(disp_img); axes[i, 2].imshow(cam, cmap='jet', alpha=0.45)
            axes[i, 2].set_title("Overlay"); axes[i, 2].axis('off')

        plt.suptitle(f"{name} — Seg-Grad-CAM (RETINOMIX-5 test samples)")
        plt.tight_layout()
        plt.savefig(os.path.join(dirs["gradcam"], f"{name}_gradcam_comparison.png"), dpi=150)
        plt.close()
        gradcam_faithfulness = float(np.mean(faithfulness_scores))
        print(f"        Grad-CAM faithfulness (Deletion-AUC): {gradcam_faithfulness:.4f}")
    except Exception as e:
        print(f"        Grad-CAM generation failed ({e}); continuing without it.")
        gradcam_faithfulness = float('nan')

    # --- Assemble + save final metrics row ---
    row = dict(model=name)
    row.update({f"internal_{k}": v for k, v in internal_metrics.items() if k != "confusion_matrix"})
    row.update({f"external_{k}": v for k, v in external_metrics.items() if k != "confusion_matrix"})
    row.update(efficiency_metrics)
    row["gradcam_faithfulness"] = gradcam_faithfulness
    row["best_threshold"] = best_thresh
    row["best_min_size"] = best_min_size

    with open(os.path.join(dirs["metrics"], f"{name}_metrics.json"), "w") as f:
        json.dump(row, f, indent=2, default=float)

    print(f"\n[{name}] Evaluation complete. Metrics saved to {dirs['metrics']}/{name}_metrics.json")
    return row


print("Evaluation harness ready: evaluate_topoeye_variant(name, model)")


## 14. RUN — train + evaluate both variants
This is the long-running cell. Leave it running overnight.

In [ ]:
# ============================================================
# RUN: TRAIN + EVALUATE BOTH TOPOEYENET VARIANTS
# ============================================================
# This is the long-running cell -- expect roughly the same per-epoch
# time as your baseline UNet/UNet++ runs (~2 min/epoch x 30 epochs x
# 2 variants + evaluation, so a few hours total; safe to leave running
# overnight). Results are checkpointed to disk after each variant, so
# if the session dies after variant 1 finishes you still have that
# variant's full metrics.json on disk.

topoeye_results = {}

# --- Variant 1: dual heads, NO structural consistency loss ---
model_noc, history_noc = train_topoeye_variant(
    "TopoEyeNet-NoConsistency", lambda_consistency=0.0
)
row_noc = evaluate_topoeye_variant("TopoEyeNet-NoConsistency", model_noc)
topoeye_results["TopoEyeNet-NoConsistency"] = row_noc

del model_noc
clear_gpu_cache()

# --- Variant 2: dual heads + structural consistency loss (full claim) ---
model_full, history_full = train_topoeye_variant(
    "TopoEyeNet-Full", lambda_consistency=cfg.LAMBDA_CONSISTENCY
)
row_full = evaluate_topoeye_variant("TopoEyeNet-Full", model_full)
topoeye_results["TopoEyeNet-Full"] = row_full

del model_full
clear_gpu_cache()

print("\n" + "=" * 60)
print("BOTH TOPOEYENET VARIANTS COMPLETE")
print("=" * 60)
for name, row in topoeye_results.items():
    print(f"{name}: internal Dice={row['internal_dice']:.4f} | "
          f"internal clDice={row['internal_cldice']:.4f} | "
          f"external(DRIVE) Dice={row['external_dice']:.4f}")


## 15. Comparison tables + how to read the ablation

In [ ]:
# ============================================================
# COMPARISON TABLES (merges with baseline results if present)
# ============================================================
# Reads every model's saved metrics.json from
# outputs/benchmarks/<Model>/metrics/ -- this includes the two
# TopoEyeNet variants just trained above, AND, if you point
# BASELINE_METRICS_DIR at the output folder from your other notebook
# (topoeye-other-models.ipynb), the previously-trained
# UNet/UNet++/DeepLabV3+/SegFormer-B0/AttentionUNet baselines too --
# so you get ONE merged table without re-running the baselines.
#
# If you don't have that folder available in this session, leave
# BASELINE_METRICS_DIR as None and you'll just get the two TopoEyeNet
# rows, which is enough to read the ablation on its own.

BASELINE_METRICS_DIR = None  # e.g. "/kaggle/working/output/benchmarks" from the other notebook's run

def _load_all_metrics(root_dirs):
    rows = []
    for root in root_dirs:
        if root is None or not os.path.exists(root):
            continue
        for name in sorted(os.listdir(root)):
            metrics_path = os.path.join(root, name, "metrics", f"{name}_metrics.json")
            if os.path.exists(metrics_path):
                with open(metrics_path) as f:
                    rows.append(json.load(f))
    return rows

all_rows = _load_all_metrics([BENCHMARK_ROOT, BASELINE_METRICS_DIR])
if not all_rows:
    print("No saved metrics.json files found yet -- run the training cell above first.")
else:
    df = pd.DataFrame(all_rows).set_index("model")

    print("\n" + "=" * 70)
    print("TABLE 1 — Internal RETINOMIX-5 Test Set")
    print("=" * 70)
    cols1 = ["internal_dice", "internal_iou", "internal_precision", "internal_recall",
             "internal_specificity", "internal_accuracy", "internal_auc_roc", "internal_cldice"]
    cols1 = [c for c in cols1 if c in df.columns]
    print(df[cols1].round(4).to_string())

    print("\n" + "=" * 70)
    print("TABLE 2 — External DRIVE Test Set (no retraining)")
    print("=" * 70)
    cols2 = ["external_dice", "external_iou", "external_precision", "external_recall",
             "external_specificity", "external_accuracy", "external_auc_roc", "external_cldice"]
    cols2 = [c for c in cols2 if c in df.columns]
    print(df[cols2].round(4).to_string())
    print("\nSanity check: external_specificity should NOT be ~0.0 and external_auc_roc")
    print("should NOT be NaN anymore now that the DRIVE ground-truth path is fixed.")
    print("If you still see those, the DRIVE dataset input path itself may differ from")
    print("the config above -- re-check cfg.DRIVE_TEST_MASK_DIR against your data source.")

    print("\n" + "=" * 70)
    print("TABLE 3 — Efficiency")
    print("=" * 70)
    cols3 = ["parameters_M", "gflops", "inference_time_ms", "fps"]
    cols3 = [c for c in cols3 if c in df.columns]
    print(df[cols3].round(3).to_string())

    print("\n" + "=" * 70)
    print("TABLE 4 — Topology (RETINOMIX-5 test)")
    print("=" * 70)
    cols4 = ["internal_cldice", "internal_betti0_error", "internal_junction_mae", "internal_endpoint_mae"]
    cols4 = [c for c in cols4 if c in df.columns]
    df4 = df[cols4].rename(columns={
        "internal_cldice": "clDice", "internal_betti0_error": "Betti-0 Error",
        "internal_junction_mae": "Junction MAE", "internal_endpoint_mae": "Endpoint MAE"})
    print(df4.round(4).to_string())

    if "internal_native_cldice" in df.columns:
        print("\n" + "=" * 70)
        print("TABLE 5 — TopoEyeNet-only: architecture-native skeleton-head check")
        print("=" * 70)
        print("(does the model's own skeleton HEAD output land inside its own")
        print(" mask HEAD output -- this is the direct novelty check, only")
        print(" meaningful for TopoEyeNet variants, not the single-output baselines)")
        native_df = df[["internal_native_cldice"]].dropna()
        print(native_df.round(4).to_string())

    df.to_csv(os.path.join(BENCHMARK_ROOT, "topoeyenet_comparison_tables.csv"))
    print(f"\nSaved merged comparison table to: "
          f"{os.path.join(BENCHMARK_ROOT, 'topoeyenet_comparison_tables.csv')}")

print("\n" + "=" * 70)
print("HOW TO READ THIS FOR THE ABLATION ARGUMENT")
print("=" * 70)
print("""
1. Compare TopoEyeNet-NoConsistency vs TopoEyeNet-Full:
   - If internal_cldice / internal_betti0_error / internal_junction_mae
     / internal_endpoint_mae improve meaningfully with the consistency
     loss ON, while internal_dice stays roughly flat, that IS your
     paper's evidence: the structural consistency term buys you
     topological correctness without costing pixel-overlap accuracy.
   - If nothing moves, the consistency loss isn't doing anything
     measurable yet and the weighting (cfg.LAMBDA_CONSISTENCY) or the
     loss formulation needs more work before you can claim it as a
     contribution.

2. Compare either TopoEyeNet variant against your baseline UNet
   (if BASELINE_METRICS_DIR is set): a competitive internal_dice
   (within ~1-2 points) plus a CLEAR win on internal_cldice /
   internal_betti0_error is a defensible IEEE Access story even
   though the backbone is a standard U-Net decoder.

3. Table 5 (native skeleton-head check) is TopoEyeNet-specific: a
   high native clDice means the skeleton head and mask head actually
   agree with each other, which is the literal thing the consistency
   loss claims to enforce -- report this number directly in the paper
   as your architecture-native ablation metric.
""")


## Notes for tomorrow morning

- **If `internal_dice` is still stuck below ~0.5** for both variants after this run, the BatchNorm/DataParallel theory was wrong (or incomplete) — the next thing to check is the skeleton target itself (is it too sparse / near-empty after resize, causing the skeleton loss term to dominate or destabilize training via `LAMBDA_SKELETON`/`LAMBDA_CONSISTENCY`). Try `lambda_skeleton=0, lambda_consistency=0` as a sanity-check variant (should behave like a plain mask-only patch U-Net and should reach ~0.8 Dice, matching your `UNet` baseline).
- **If `external_specificity` is still ~0.0** after this run, the DRIVE fix in Cell 3 didn't take — check the sanity-check print in Cell 3 (vessel pixel % should be single digits to low teens, not 40%+).
- Everything is saved under `outputs/benchmarks/TopoEyeNet-NoConsistency/` and `outputs/benchmarks/TopoEyeNet-Full/` (weights, history, metrics.json, predictions, confusion matrices, Grad-CAM figures) — nothing here depends on kernel state surviving until morning.
